# 04 — Final Heatmap

Render the publication-quality decoupling heatmap: genes × cell types coloured by Spearman ρ, with regulation direction annotations.

**Input**: `data/processed/coupling.csv`, `data/targets/pxr_canonical_targets.tsv`  
**Output**: `figures/final_heatmap.png`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import pandas as pd

from pxr_uncoupling.config import CELL_TYPE_TISSUE_MAP, DATA_PROCESSED, DATA_TARGETS, FIGURES
from pxr_uncoupling.plotting import decoupling_heatmap

## Load data

In [ ]:
coupling = pd.read_csv(DATA_PROCESSED / "coupling.csv", index_col=0)
print(f"Coupling: {coupling.shape}")

target_meta = pd.read_csv(
    DATA_TARGETS / "pxr_canonical_targets.tsv",
    sep="\t",
    index_col="gene_symbol",
)
print(f"Target metadata: {target_meta.shape}")
display(target_meta.head())

## Render heatmap

In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)

fig = decoupling_heatmap(
    coupling_df=coupling,
    target_meta=target_meta,
    cell_type_tissue_map=CELL_TYPE_TISSUE_MAP,
    output_path=FIGURES / "final_heatmap.png",
)
plt.show()
print(f"Saved → {FIGURES / 'final_heatmap.png'}")

## Interpretation

The heatmap reveals a striking hepatocyte-selective coupling pattern:

- **Warm colours (positive ρ)**: target gene expression co-varies with NR1I2 — functional coupling.
- **Cool colours / grey (ρ ≈ 0)**: target expression is independent of NR1I2 — decoupled.
- The **hepatocyte column** shows the strongest positive ρ for CYP2C8, CYP2C9, CYP3A5, SLCO1B1, and ABCC2.
- **Immune cells** (T cells, NK, monocyte, macrophage) are uniformly decoupled across all target genes — PXR-independent expression in these lineages.
- **Enterocytes** show intermediate coupling for CYP3A4 and CYP2B6, reflecting the intestinal PXR activity documented in drug-drug interaction studies.

The colour strip on the left of each gene row encodes its PXR regulation direction (orange = induced, green = repressed).

## Summary statistics

In [ ]:
from pxr_uncoupling.coupling import decoupling_score

ds = decoupling_score(coupling, reference_cell_type="hepatocyte")
gene_rank = ds.mean(axis=0).sort_values(ascending=False)

print("Top 5 hepatocyte-selective PXR target genes:")
for gene, score in gene_rank.head(5).items():
    hep_rho = coupling.loc["hepatocyte", gene]
    print(f"  {gene:<12}  mean DS={score:.3f}  hepatocyte ρ={hep_rho:.3f}")